# Test ingestion + retrieval endpoints

Start the service first:

```powershell
uv run python main.py
```

Then run the cells below.

In [1]:
import json
import requests

BASE_URL = "http://localhost:9000"

def pretty(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))

## Health

In [2]:
r = requests.get(f"{BASE_URL}/health", timeout=10)
r.raise_for_status()
pretty(r.json())

{
  "status": "ok"
}


## Ingest a YouTube video

First call: pulls metadata + transcript, chunks, embeds, upserts into Qdrant.
Second call with the same URL should return `already_indexed=true` and `chunks_indexed=0`.

In [11]:
VIDEO_1 = "https://www.youtube.com/watch?v=gaa_5m3cmSc"
VIDEO_2 = "https://www.youtube.com/watch?v=CuOuU6XO7zk"

r = requests.post(
    f"{BASE_URL}/ingest",
    json={"urls": [VIDEO_1, VIDEO_2]},
    timeout=60,
)
r.raise_for_status()
ingest_result = r.json()
pretty(ingest_result)

{
  "results": [
    {
      "url": "https://www.youtube.com/watch?v=gaa_5m3cmSc",
      "success": true,
      "video_id": "gaa_5m3cmSc",
      "title": "The Irishman - Al Pacino Says You're Late Clip | Netflix",
      "channel": "Still Watching Netflix",
      "chunks_indexed": 0,
      "already_indexed": true,
      "error": null
    },
    {
      "url": "https://www.youtube.com/watch?v=CuOuU6XO7zk",
      "success": true,
      "video_id": "CuOuU6XO7zk",
      "title": "The Wolf of Wall Street: Jordan Belfort's Introduction to Wall Street",
      "channel": "Binge Society",
      "chunks_indexed": 13,
      "already_indexed": false,
      "error": null
    }
  ]
}


## Retrieve top-k chunks

In [13]:
QUERY = "When he talks about decimal points and high frequencies"

r = requests.post(
    f"{BASE_URL}/retrieve",
    json={"query": QUERY, "top_k": 2},
    timeout=60,
)
r.raise_for_status()
retrieval = r.json()
pretty(retrieval)

{
  "query": "When he talks about decimal points and high frequencies",
  "results": [
    {
      "score": 0.57026064,
      "text": "this racket. I myself, I jerk off at least twice a day. Wow. >> Once in the morning, right after I work out, and then once right after lunch. Really? Mhm. All right. I want to. That's not why I do it. I do it cuz I [ __ ] need to. Think about it. You're dealing with numbers all day long. Decimal points, high frequencies, bang bang bang. [ __ ] digits all very acidic above the shoulders, mustard [ __ ] Mhm. All right. Mhm. It's kind of going to wake some people up. Mhm. Right. You got to feed the geese to keep the blood flowing. I keep the rhythm below the belt. Done. This is not a tip. This is a prescription. Trust me. Mhm. If you don't, you will fall out of balance, split your differential, and tip the [ __ ] over, or wear [ __ ] I've seen this happen. Implode",
      "metadata": {
        "start_time": 340.84,
        "end_time": 449.84,
        "vide

In [ ]:
# Compact view: score, time range, snippet
for i, hit in enumerate(retrieval["results"], start=1):
    md = hit["metadata"]
    snippet = hit["text"][:200].replace("\n", " ")
    print(f"#{i}  score={hit['score']:.4f}  [{md.get('start_time')}-{md.get('end_time')}]  {md.get('title')}")
    print(f"     {snippet}...\n")

## Try a few more queries

In [ ]:
queries = [
    "what is the main topic of the video",
    "any mention of performance or optimization",
    "introduction or opening hook",
]

for q in queries:
    r = requests.post(f"{BASE_URL}/retrieve", json={"query": q, "top_k": 3}, timeout=60)
    r.raise_for_status()
    res = r.json()["results"]
    print(f"\n=== {q} ===")
    for i, hit in enumerate(res, start=1):
        md = hit["metadata"]
        snippet = hit["text"][:160].replace("\n", " ")
        print(f"  #{i} score={hit['score']:.4f} [{md.get('start_time')}-{md.get('end_time')}] {snippet}...")